[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/motion_plane.ipynb)

# Motion in a plane full course: the practice set, in Python

Twenty-eight exercises in three tiers, every answer checked: **tier 1** warm-ups (one idea, one
step), **tier 2** core problems (resultants, components, the dot and cross products, motion with constant
acceleration, projectiles and circles, several steps each) and **tier 3** stretch problems (work backwards, spot the
trap, combine ideas, "what changes if"). Each tier is harder than the last.

Under every tier there's a short **how you'd do this in Python** cell: the same calculations with Python's `math`
module, so you can run them on your own numbers. And the course's **simulations**, which you move yourself: resolve a
vector, launch a ball at any speed and angle, push one phone off a table while another drops, and drive round a
roundabout. Every simulation steps the motion forward a millisecond at a time from
nothing but the acceleration, and every one is checked against the course's formulas.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Two decimals
are enough, in the unit the question names; angles in degrees. Units are fine to type (`35.35 m`, `30°`, `4 m/s²`).
You can type the arithmetic too: `sqrt(300**2+400**2)` or `2*pi/40`. For several answers, separate them with commas,
in the order the question lists them (the box's label shows the order). You can also put your answer between the
quotes in a cell's last line, `practice("P1.2", "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every formula, edge test and example from the course on two pages:
[motion in a plane cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/motion_plane_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the simulator and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # bodies and paths in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# one colour per quantity, as in the course: velocity SKY · acceleration MINT · force CORAL · energy VIOLET ·
# bodies, paths and displacement CHALK · the one thing to look at GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="simulated · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- the course's simulator (a copy of the series' own: fourth-order Runge–Kutta, fixed steps) ----
g = 9.8                                               # m/s², as in the course

def simulate(deriv, state, t_end, dt=1e-3, stop=None):
    """Fixed-step fourth-order Runge–Kutta. deriv(t, s) -> ds/dt as a tuple; state a tuple of floats. Returns the list of
    (t, state) samples from 0 to t_end (inclusive), or up to the first step where stop(t, s) is True, refined by linear
    interpolation so the last sample sits on the event (a projectile's landing: stop=lambda t, s: s[1] < 0)."""
    t, s = 0.0, tuple(float(v) for v in state)
    out = [(t, s)]
    n = int(round(t_end / dt))
    for _ in range(n):
        k1 = deriv(t, s)
        k2 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k1)))
        k3 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k2)))
        k4 = deriv(t + dt, tuple(a + dt * b for a, b in zip(s, k3)))
        s2 = tuple(a + dt / 6 * (b1 + 2 * b2 + 2 * b3 + b4) for a, b1, b2, b3, b4 in zip(s, k1, k2, k3, k4))
        t2 = t + dt
        if stop is not None and stop(t2, s2):
            i = getattr(stop, "var", None)
            if i is not None and s2[i] != s[i]:
                f = (getattr(stop, "level", 0.0) - s[i]) / (s2[i] - s[i])
                t2 = t + f * dt
                s2 = tuple(a + f * (b - a) for a, b in zip(s, s2))
            out.append((t2, s2))
            return out
        t, s = t2, s2
        out.append((t, s))
    return out

def close(a, b, rel=1e-4, abs_=1e-9):
    """True when a simulated value matches the closed form (relative 1e-4 by default)."""
    return abs(a - b) <= max(rel * max(abs(a), abs(b)), abs_)

def fly(u, th, h=0.0, dt=1e-3, t_end=20.0):
    """A ball launched from height h at speed u (m/s), angle θ (degrees), no air: samples (t, (x, y, vx, vy)) to the
    landing. The only physics in it: the acceleration is (0, −g)."""
    stop = lambda t, s: s[1] < 0; stop.var, stop.level = 1, 0.0
    th = math.radians(th)
    return simulate(lambda t, s: (s[2], s[3], 0.0, -g), (0.0, h, u * math.cos(th), u * math.sin(th)), t_end, dt, stop)

def circle_run(r, v, turns=1.0, dt=1e-3):
    """A point started at (r, 0) moving at v along +y, given only an acceleration v²/r towards the centre."""
    w2 = (v / r) ** 2
    t_end = turns * 2 * math.pi * r / v
    n = math.ceil(t_end / dt)
    return simulate(lambda t, s: (s[2], s[3], -w2 * s[0], -w2 * s[1]), (r, 0.0, 0.0, v), t_end, t_end / n)

# the course's formulas (closed forms), for the simulations to be checked against
def t_flight(u, th): return 2 * u * math.sin(math.radians(th)) / g          # T = 2u sin θ / g
def h_max(u, th): return (u * math.sin(math.radians(th))) ** 2 / (2 * g)    # H = u² sin²θ / 2g
def r_range(u, th): return u * u * math.sin(math.radians(2 * th)) / g        # R = u² sin 2θ / g
def reach_from_height(u, th, h):
    """Where (x) and when (t) a ball launched h above level ground at u, θ comes down: the larger root of
    h + v_y t − ½ g t² = 0, then x = v_x t."""
    vx, vy = u * math.cos(math.radians(th)), u * math.sin(math.radians(th))
    t = (vy + math.sqrt(vy * vy + 2 * g * h)) / g
    return vx * t, t

# the simulator against the formulas: the course's kick, 20 m/s at 30°
_kick = fly(20, 30)
assert close(_kick[-1][0], t_flight(20, 30)) and abs(_kick[-1][1][0] - r_range(20, 30)) < 1e-3
assert abs(max(s[1] for _, s in _kick) - h_max(20, 30)) < 1e-4
assert round(r_range(20, 30), 2) == 35.35 and round(t_flight(20, 30), 2) == 2.04 and round(h_max(20, 30), 2) == 5.10

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets, sqrt(…) and pi."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id == "pi": return math.pi
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "sqrt" and len(n.args) == 1: return math.sqrt(ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNITS = r"(m/s²|m/s\^2|m/s2|m/s|rad/s|km/h|degrees?|°|(?<![a-z])(deg|km|m|s|g)\b)"

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-")
    t = re.sub(UNITS, "", s, flags=re.I).replace("×", "*").replace("÷", "/").replace("^", "**").replace("π", "pi").strip()
    try: return calc(t)
    except Exception: return " ".join(s.lower().split())

def right(got, want, tol):
    if isinstance(want, str): return isinstance(got, str) and got == want.lower()
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    if isinstance(v, str): return v
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (episodes/ph03_motion_plane/practice.py's questions and answers) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Scalars and vectors',
     "q": 'Which of these are vectors: (1) mass 3 kg, (2) a velocity of 12 m/s north, (3) a temperature of 20 °C, (4) a displacement of 5 m east? Give the item numbers in order as a two-digit number.',
     "parts": ["the vectors' item numbers, in order"],
     "answer": [24], "tol": [0], "function": '—',
     "working": 'Velocity and displacement have a size and a direction (items 2 and 4); mass and temperature are scalars'},
    {"id": 'P1.2', "tier": 1, "group": 'Scalars and vectors',
     "q": "A runner goes 300 m east, then 400 m north. What is her displacement's magnitude, in metres?",
     "parts": ['displacement (m)'],
     "answer": [500.0], "tol": [0.005], "function": 'SQRT',
     "working": '√(300² + 400²) = √250,000 = 500 m (she ran 700 m)'},
    {"id": 'P1.3', "tier": 1, "group": 'Scalars and vectors',
     "q": 'A scooter moves at 6 m/s west for 5 s. What is its displacement (magnitude in metres)?',
     "parts": ['displacement (m, west)'],
     "answer": [30.0], "tol": [0.005], "function": '—',
     "working": 'displacement = velocity × time = 5 × (6 m/s west) = 30 m west'},
    {"id": 'P1.4', "tier": 1, "group": 'Components and the products',
     "q": 'Resolve a velocity of 12 m/s at 25° above the x axis: give v x and v y in m/s.',
     "parts": ['v x (m/s)', 'v y (m/s)'],
     "answer": [10.875693444439799, 5.071419140888393], "tol": [0.005, 0.005], "function": 'COS, SIN, RADIANS',
     "working": '12 cos 25° = 10.8757; 12 sin 25° = 5.0714'},
    {"id": 'P1.5', "tier": 1, "group": 'Components and the products',
     "q": 'What is the unit vector along 5 i + 12 j? Give its x and y components.',
     "parts": ['x component of the unit vector', 'y component of the unit vector'],
     "answer": [0.38461538461538464, 0.9230769230769231], "tol": [0.005, 0.005], "function": 'SQRT',
     "working": '|A| = 13, so Â = (5/13) i + (12/13) j = 0.3846 i + 0.9231 j'},
    {"id": 'P1.6', "tier": 1, "group": 'Components and the products',
     "q": 'Find A · B for A = 4 i + 1 j and B = 2 i − 3 j.',
     "parts": ['A · B'],
     "answer": [5], "tol": [0.005], "function": '—',
     "working": '4 × 2 + 1 × (−3) = 8 − 3 = 5'},
    {"id": 'P1.7', "tier": 1, "group": 'Motion in a plane',
     "q": 'A puck slides at a constant velocity of 2 i + 1 j m/s from the origin. Where is it after 6 s? Give x and y in metres.',
     "parts": ['x (m)', 'y (m)'],
     "answer": [12.0, 6.0], "tol": [0.005, 0.005], "function": '—',
     "working": 'r = v t = 6 × (2, 1) = (12, 6) m'},
    {"id": 'P1.8', "tier": 1, "group": 'Projectile motion',
     "q": 'A stone is pushed horizontally off a wall 4.9 m high. How long does it take to land, in seconds?',
     "parts": ['time to land (s)'],
     "answer": [1.0], "tol": [0.005], "function": 'SQRT',
     "working": "4.9 = ½ × 9.8 × t² → t² = 1 → t = 1.00 s (the push across doesn't matter)"},
    {"id": 'P1.9', "tier": 1, "group": 'Projectile motion',
     "q": 'A ball is launched at 10 m/s at 45° on level ground. What is its range, in metres?',
     "parts": ['range R (m)'],
     "answer": [10.204081632653061], "tol": [0.005], "function": 'SIN, RADIANS',
     "working": 'R = u² sin 2θ / g = 100 × sin 90° / 9.8 = 10.2041 m'},
    {"id": 'P1.10', "tier": 1, "group": 'Uniform circular motion',
     "q": "A wheel's rim point moves at 3 m/s on a circle of radius 0.5 m. What is its centripetal acceleration, in m/s²?",
     "parts": ['centripetal acceleration (m/s²)'],
     "answer": [18.0], "tol": [0.005], "function": '—',
     "working": 'v²/r = 9/0.5 = 18 m/s², towards the centre'},
    {"id": 'P2.1', "tier": 2, "group": 'Scalars and vectors',
     "q": "Two displacements of 5 m and 8 m have 60° between them, tail to tail. Find the resultant's magnitude (m) and its angle from the 5 m one (degrees).",
     "parts": ['resultant (m)', 'angle from the 5 m one (degrees)'],
     "answer": [11.357816691600547, 37.58908946897495], "tol": [0.005, 0.005], "function": 'SQRT, COS, ATAN',
     "working": 'R² = 25 + 64 + 2 × 5 × 8 × 0.5 = 129 → R = 11.3578; tan α = 8 sin 60 / (5 + 8 cos 60) → α = 37.5891°'},
    {"id": 'P2.2', "tier": 2, "group": 'Scalars and vectors',
     "q": "A car's velocity changes from 15 m/s east to 15 m/s south. What is the magnitude of the change in velocity, in m/s?",
     "parts": ['|Δv| (m/s)'],
     "answer": [21.213203435596427], "tol": [0.005], "function": 'SQRT',
     "working": 'Δv = v₂ − v₁ = (0, −15) − (15, 0) = (−15, −15): |Δv| = 21.2132 m/s, pointing south-west'},
    {"id": 'P2.3', "tier": 2, "group": 'Components and the products',
     "q": "Add A = 6 m at 0° and B = 4 m at 120° by components. Give the resultant's x and y components and its magnitude.",
     "parts": ['R x (m)', 'R y (m)', '|R| (m)'],
     "answer": [4.000000000000001, 3.464101615137755, 5.291502622129182], "tol": [0.005, 0.005, 0.005], "function": 'COS, SIN, SQRT',
     "working": 'B = (4 cos 120, 4 sin 120) = (−2, 3.4641); sum = (4, 3.4641); |R| = 5.2915'},
    {"id": 'P2.4', "tier": 2, "group": 'Components and the products',
     "q": 'Find the angle between A = 1 i + 2 j and B = 3 i − 1 j, in degrees.',
     "parts": ['angle between them (degrees)'],
     "answer": [81.86989764584403], "tol": [0.005], "function": 'ACOS, DEGREES',
     "working": 'A · B = 3 − 2 = 1; |A| = √5, |B| = √10; cos θ = 1/√50 → θ = 81.8699°'},
    {"id": 'P2.5', "tier": 2, "group": 'Components and the products',
     "q": 'Find the k component of A × B for A = 2 i + 3 j and B = 4 i + 1 j, and the area of the parallelogram they span.',
     "parts": ['k component of A × B', "the parallelogram's area"],
     "answer": [-10, 10], "tol": [0.005, 0.005], "function": '—',
     "working": 'A x B y − A y B x = 2 × 1 − 3 × 4 = −10: A × B = −10 k (into the page); area 10'},
    {"id": 'P2.6', "tier": 2, "group": 'Motion in a plane',
     "q": 'A boat starts at the origin with velocity 2 i m/s and a constant acceleration 0.5 i + 1 j m/s². Find its velocity and position after 4 s (x and y of each).',
     "parts": ['v x after 4 s (m/s)', 'v y after 4 s (m/s)', 'x after 4 s (m)', 'y after 4 s (m)'],
     "answer": [4.0, 4.0, 12.0, 8.0], "tol": [0.005, 0.005, 0.005, 0.005], "function": '—',
     "working": 'v = u + a t = (2 + 2, 0 + 4) = (4, 4); r = u t + ½ a t² = (8 + 4, 0 + 8) = (12, 8)'},
    {"id": 'P2.7', "tier": 2, "group": 'Projectile motion',
     "q": 'A ball rolls off a table 0.9 m high at 3 m/s. How far from the table does it land (m), and how fast is it moving then (m/s)?',
     "parts": ['distance from the table (m)', 'speed at landing (m/s)'],
     "answer": [1.2857142857142856, 5.1613951602255765], "tol": [0.005, 0.005], "function": 'SQRT',
     "working": 't = √(2 × 0.9/9.8) = 0.4286 s; x = 3t = 1.2857 m; v y = g t = 4.2000; speed = √(9 + v y²) = 5.1614 m/s'},
    {"id": 'P2.8', "tier": 2, "group": 'Projectile motion',
     "q": 'A ball is kicked at 18 m/s at 50°. Find its time of flight (s), maximum height (m) and range (m).',
     "parts": ['time of flight T (s)', 'maximum height H (m)', 'range R (m)'],
     "answer": [2.81404081145747, 9.700561468471566, 32.5589502016281], "tol": [0.005, 0.005, 0.005], "function": 'SIN, RADIANS',
     "working": 'T = 2 × 18 sin 50 / 9.8 = 2.8140; H = (18 sin 50)²/19.6 = 9.7006; R = 324 sin 100 / 9.8 = 32.5590'},
    {"id": 'P2.9', "tier": 2, "group": 'Uniform circular motion',
     "q": "A playground roundabout of radius 2 m turns once every 4 s. Find ω (rad/s), the rim speed (m/s) and the rim's centripetal acceleration (m/s²).",
     "parts": ['ω (rad/s)', 'rim speed v (m/s)', 'centripetal acceleration (m/s²)'],
     "answer": [1.5707963267948966, 3.141592653589793, 4.934802200544679], "tol": [0.005, 0.005, 0.005], "function": 'PI',
     "working": 'ω = 2π/4 = 1.5708; v = ωr = 3.1416; a = v²/r = 4.9348'},
    {"id": 'P2.10', "tier": 2, "group": 'Uniform circular motion',
     "q": 'A washing-machine drum of radius 0.25 m spins at 1,200 turns a minute. What is the centripetal acceleration of its wall (m/s²), and how many times g is that?',
     "parts": ['centripetal acceleration (m/s²)', 'how many times g'],
     "answer": [3947.8417604357433, 402.8409959628309], "tol": [0.005, 0.005], "function": 'PI',
     "working": 'ω = 1,200 × 2π/60 = 125.6637 rad/s; a = ω² r = 3947.8418 m/s², 402.8410 g'},
    {"id": 'P3.1', "tier": 3, "group": 'Scalars and vectors',
     "q": 'A 7 m and a 4 m displacement give a resultant of 5 m. What angle lies between them, tail to tail (degrees)?',
     "parts": ['angle between them (degrees)'],
     "answer": [135.58469140280704], "tol": [0.005], "function": 'ACOS, DEGREES',
     "working": 'cos θ = (25 − 49 − 16)/56 = −40/56 → θ = 135.5847°'},
    {"id": 'P3.2', "tier": 3, "group": 'Components and the products',
     "q": "A velocity has components v x = −5 m/s and v y = −5 m/s. A calculator's inverse tangent of v y/v x gives 45°. What is the true direction, in degrees from the +x axis between 0 and 360?",
     "parts": ['true direction (degrees from +x, 0 to 360)'],
     "answer": [225.0], "tol": [0.005], "function": 'ATAN2, DEGREES',
     "working": 'The arrow points left and down (third quadrant): 45° + 180° = 225°. atan2(−5, −5) gives −135° = 225°'},
    {"id": 'P3.3', "tier": 3, "group": 'Components and the products',
     "q": 'Show the dot and cross products agree on the angle for A = 3 i + 4 j and B = 4 i + 3 j: give A · B, the k component of A × B, and the angle (degrees).',
     "parts": ['A · B', 'k component of A × B', 'angle (degrees)'],
     "answer": [24, -7, 16.260204708311967], "tol": [0.005, 0.005, 0.005], "function": 'ACOS, ATAN',
     "working": 'A · B = 24 = 25 cos θ; A × B = 9 − 16 = −7 = −25 sin θ; tan θ = 7/24 → θ = 16.2602°'},
    {"id": 'P3.4', "tier": 3, "group": 'Motion in a plane',
     "q": 'A drone leaves the origin at 4 i m/s. After 5 s it is at 25 i + 10 j m. Find its constant acceleration (x and y, m/s²).',
     "parts": ['a x (m/s²)', 'a y (m/s²)'],
     "answer": [0.4, 0.8], "tol": [0.005, 0.005], "function": '—',
     "working": 'x: 25 = 20 + ½ a x × 25 → a x = 0.4; y: 10 = ½ a y × 25 → a y = 0.8'},
    {"id": 'P3.5', "tier": 3, "group": 'Projectile motion',
     "q": 'A long pass must land 50 m away, kicked at 30°. What launch speed does it need (m/s)?',
     "parts": ['launch speed (m/s)'],
     "answer": [23.78661942785972], "tol": [0.005], "function": 'SQRT, SIN',
     "working": 'u = √(R g / sin 2θ) = √(490/sin 60) = 23.7866 m/s'},
    {"id": 'P3.6', "tier": 3, "group": 'Projectile motion',
     "q": 'A hose throws water at 12 m/s. At which two angles does it land 12 m away on level ground (degrees)? Which spends less time in the air?',
     "parts": ['the low angle (degrees)', 'the high angle (degrees)'],
     "answer": [27.376246503973704, 62.623753496026296], "tol": [0.005, 0.005], "function": 'ASIN, DEGREES',
     "working": 'sin 2θ = 12 × 9.8/144 = 0.8167 → θ = 27.3762° or 62.6238° (they add up to 90°); the low one is quicker'},
    {"id": 'P3.7', "tier": 3, "group": 'Projectile motion',
     "q": 'A ball is thrown at 15 m/s at 35° from a balcony 6 m above level ground. Where does it land (m from the balcony)? Why is the range formula wrong here, and by how much?',
     "parts": ['where it lands (m from the balcony)', 'how far the range formula falls short (m)'],
     "answer": [28.143430701274106, 6.568855224046619], "tol": [0.005, 0.005], "function": 'SQRT, TAN',
     "working": 'Solve 0 = 6 + x tan 35 − 9.8 x²/(2 × 225 cos²35): x = 28.1434 m. The range formula assumes landing at launch height: 21.5746 m, short by 6.5689'},
    {"id": 'P3.8', "tier": 3, "group": 'Uniform circular motion',
     "q": "On a curve of radius 80 m, a car's centripetal acceleration must stay below 5 m/s². What is its top speed (m/s)? If the curve were half as tight (radius 160 m), what would the top speed be?",
     "parts": ['top speed at 80 m (m/s)', 'top speed at 160 m (m/s)'],
     "answer": [20.0, 28.284271247461902], "tol": [0.005, 0.005], "function": 'SQRT',
     "working": 'v = √(a r) = √400 = 20 m/s; at 160 m, √800 = 28.2843 m/s (√2 times, not twice)'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def split(text, q):
    return [read_part(p) for p in re.split(r"[,;]", text) if p.strip()]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = split(box.value, q)
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            if q["function"] not in ("—", ""):
                fns = ", ".join("math." + f_.strip().lower() for f_ in q["function"].split(","))
                print(textwrap.fill(f"In Python: {fns}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

# the checker itself: a right answer passes, a wrong one doesn't, with or without units
assert right(read_part("sqrt(300**2+400**2)"), 500, 0.005) and not right(read_part("700"), 500, 0.005)
assert read_part("35.35 m") == 35.35 and read_part("30°") == 30 and read_part("4 m/s²") == 4 and read_part("0.4 rad/s") == 0.4
assert read_part("35.35m") == 35.35 and read_part("403 g") == 403 and abs(read_part("sqrt(2)") - math.sqrt(2)) < 1e-12
assert abs(read_part("2*pi/40") - 2 * math.pi / 40) < 1e-12 and read_part("−12.44") == -12.44

## Tier 1 · Warm-up

One idea, one step: which quantities are vectors, a displacement from two legs, a displacement from a velocity,
resolving a velocity, a unit vector, a dot product, a position at constant velocity, a fall time, a range and a
centripetal acceleration.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Python's `math` module measures angles in **radians**: `math.cos(30)` is the cosine of 30
radians. Wrap degrees in `math.radians(…)`, and turn an inverse function's answer back with `math.degrees(…)`.
`math.hypot(x, y)` is √(x² + y²). And one trap to know: Python's `math.atan2` takes **y first, then x**, the
other way round from how you'd say "x and y". Each line below computes one warm-up answer and checks it against the
practice set.

In [ ]:
d12 = math.hypot(300, 400)                                       # P1.2: the displacement, not the 700 m she ran
d13 = 6 * 5                                                      # P1.3: velocity × time
vx14, vy14 = 12 * math.cos(math.radians(25)), 12 * math.sin(math.radians(25))   # P1.4: resolve
a15 = (5, 12); u15 = (a15[0] / math.hypot(*a15), a15[1] / math.hypot(*a15))      # P1.5: divide by the length
dot16 = 4 * 2 + 1 * (-3)                                         # P1.6: A x B x + A y B y
r17 = (2 * 6, 1 * 6)                                             # P1.7: r = v t
t18 = math.sqrt(2 * 4.9 / g)                                     # P1.8: the push across doesn't matter
R19 = 10 ** 2 * math.sin(math.radians(2 * 45)) / g               # P1.9: R = u² sin 2θ / g
a110 = 3 ** 2 / 0.5                                              # P1.10: v²/r

print(f"displacement {d12} m   scooter {d13} m   v x {vx14:.4f}, v y {vy14:.4f} m/s   unit vector "
      f"({u15[0]:.4f}, {u15[1]:.4f})   A · B {dot16}")
print(f"puck at {r17} m   fall {t18:.4f} s   range {R19:.4f} m   centripetal {a110} m/s²")
print("the radians trap:", round(20 * math.cos(30), 2), "instead of", round(20 * math.cos(math.radians(30)), 2))
for k, v in (("P1.2", [d12]), ("P1.3", [d13]), ("P1.4", [vx14, vy14]), ("P1.5", list(u15)), ("P1.6", [dot16]),
             ("P1.7", list(r17)), ("P1.8", [t18]), ("P1.9", [R19]), ("P1.10", [a110])):
    assert agrees(k, v), k
assert round(20 * math.cos(30), 2) == 3.09                       # the video's wrong line

### Resolve a vector: slide the angle

The course's kick: 20 m/s at 30° splits into 17.32 m/s across and 10.00 m/s up. Slide the angle and watch the share
change: the across part shrinks as the up part grows, and the two parts always rebuild the length,
√(across² + up²) = 20.

In [ ]:
def resolve_picture(size=20.0, angle=30.0):
    ax_, ay_ = size * math.cos(math.radians(angle)), size * math.sin(math.radians(angle))
    fig, ax = plt.subplots(figsize=(10, 7.4))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.9, bottom=0.09)
    kw = dict(angles="xy", scale_units="xy", scale=1, width=0.006)
    ax.quiver([0], [0], [ax_], [0], color=SKY, alpha=0.55, **kw)
    ax.quiver([ax_], [0], [0], [ay_], color=SKY, alpha=0.55, **kw)
    ax.quiver([0], [0], [ax_], [ay_], color=GOLD, **kw)
    ax.plot([0, ax_], [ay_, ay_], color=DIM, ls=":", lw=1.2); ax.plot([ax_, ax_], [0, ay_], color=DIM, ls=":", lw=1.2)
    ax.text(ax_ / 2, -1.6, f"across {ax_:.2f} m/s", color=SKY, ha="center")
    ax.text(ax_ + 0.6, ay_ / 2, f"up {ay_:.2f} m/s", color=SKY, va="center")
    ax.set_xlim(-2, 34); ax.set_ylim(-3, 27); ax.set_aspect("equal"); ax.set_xlabel("across (m/s)"); ax.set_ylabel("up (m/s)")
    fig.suptitle(f"{size:g} m/s at {angle:g}°: {size:g} cos θ = {ax_:.2f} across, {size:g} sin θ = {ay_:.2f} up; "
                 f"√(across² + up²) = {math.hypot(ax_, ay_):.2f}", x=0.08, ha="left", fontsize=13, color=GOLD)
    plt.show()

assert round(20 * math.cos(math.radians(30)), 2) == 17.32 and round(20 * math.sin(math.radians(30)), 9) == 10

In [ ]:
interact(resolve_picture, size=FloatSlider(value=20, min=5, max=25, step=1, description="size (m/s)"),
         angle=FloatSlider(value=30, min=0, max=90, step=1, description="angle (°)"));

## Tier 2 · Core

The course's standard problems, several steps each: a resultant by the law of cosines, a change in velocity,
adding by components, the angle from the dot product, the cross product and its area, motion with a constant
acceleration, a ball off a table, a full projectile, a roundabout and a spinning drum.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**Vectors** are pairs `(x, y)`: add them part by part, the dot product is `ax*bx + ay*by`, and the cross product's
k component is `ax*by - ay*bx`. **The angle between two vectors** comes from the dot product:
`math.degrees(math.acos(dot / (|A| |B|)))`. **Motion with a constant acceleration** works on x and y separately:
v = u + a t and r = u t + ½ a t². **Projectiles**: T = 2u sin θ / g, H = (u sin θ)² / 2g, R = u² sin 2θ / g.
**Circles**: ω = 2π / T, v = ω r, a = v² / r = ω² r.

In [ ]:
def polar(m, th): return (m * math.cos(math.radians(th)), m * math.sin(math.radians(th)))
def mag(v): return math.hypot(v[0], v[1])
def dot(a, b): return a[0] * b[0] + a[1] * b[1]
def cross(a, b): return a[0] * b[1] - a[1] * b[0]
def angle_between(a, b): return math.degrees(math.acos(dot(a, b) / (mag(a) * mag(b))))

R21 = math.sqrt(5 ** 2 + 8 ** 2 + 2 * 5 * 8 * math.cos(math.radians(60)))               # P2.1: law of cosines
al21 = math.degrees(math.atan2(8 * math.sin(math.radians(60)), 5 + 8 * math.cos(math.radians(60))))
assert agrees("P2.1", [R21, al21])
dv = (0 - 15, -15 - 0)                                                                   # P2.2: v₂ − v₁
assert agrees("P2.2", [mag(dv)])
s23 = tuple(a + b for a, b in zip(polar(6, 0), polar(4, 120)))                           # P2.3: by components
assert agrees("P2.3", [*s23, mag(s23)])
assert agrees("P2.4", [angle_between((1, 2), (3, -1))])                                  # P2.4
c25 = cross((2, 3), (4, 1)); assert agrees("P2.5", [c25, abs(c25)])                      # P2.5: −10 k, into the page
u, a, t = (2, 0), (0.5, 1), 4                                                            # P2.6: x and y separately
v26 = tuple(ui + ai * t for ui, ai in zip(u, a)); r26 = tuple(ui * t + 0.5 * ai * t * t for ui, ai in zip(u, a))
assert agrees("P2.6", [*v26, *r26])
t27 = math.sqrt(2 * 0.9 / g)                                                             # P2.7: off the table
assert agrees("P2.7", [3 * t27, math.hypot(3, g * t27)])
assert agrees("P2.8", [t_flight(18, 50), h_max(18, 50), r_range(18, 50)])                # P2.8
w29 = 2 * math.pi / 4; assert agrees("P2.9", [w29, w29 * 2, (w29 * 2) ** 2 / 2])          # P2.9
w210 = 1200 * 2 * math.pi / 60; a210 = w210 ** 2 * 0.25                                   # P2.10: turns a minute → rad/s
assert agrees("P2.10", [a210, a210 / g])
print(f"P2.1 R {R21:.4f} m at {al21:.4f}°   P2.3 sum ({s23[0]:.4f}, {s23[1]:.4f}), length {mag(s23):.4f} m")
print(f"P2.6 v {v26} m/s, r {r26} m   P2.8 T {t_flight(18, 50):.4f} s, H {h_max(18, 50):.4f} m, R {r_range(18, 50):.4f} m")
print(f"P2.10 the drum's wall: {a210:.1f} m/s², {a210 / g:.0f} times g")

### The kick, row by row

The course's kick, 20 m/s at 30°, simulated a millisecond at a time and printed every 0.1 s. v x never changes; v y falls by 0.98 m/s every 0.1 s. The last row is the landing: after
2.04 s, 35.35 m away, with v y back to −10 m/s.

In [ ]:
rows = [s for t_, s in _kick[::100]] + [_kick[-1][1]]
times = [round(t_, 3) for t_, s in _kick[::100]] + [round(_kick[-1][0], 3)]
table = pd.DataFrame(rows, columns=["x (m)", "y (m)", "v x (m/s)", "v y (m/s)"], index=pd.Index(times, name="t (s)"))
print(table.round(3).to_string())
assert round(table.index[-1], 2) == 2.04 and round(table["x (m)"].iloc[-1], 2) == 35.35 and round(table["v y (m/s)"].iloc[-1], 2) == -10

### Launch a ball: slide the speed and the angle

The ball below is simulated from nothing but its starting velocity and the downward acceleration g. The title
compares where it lands with the course's formulas: the time of flight T = 2u sin θ / g, the maximum height
H = (u sin θ)² / 2g and the range R = u² sin 2θ / g. The dashed path is the twin angle, 90° − θ: it lands on the
same spot. Try 45° (the farthest), then 30° and 60°.

In [ ]:
def launch_picture(speed=20.0, angle=30.0):
    run = fly(speed, angle)
    t_end, (x_end, *_) = run[-1]
    xs, ys = [s[0] for _, s in run], [s[1] for _, s in run]
    twin = fly(speed, 90 - angle) if 0 < angle < 90 else run
    fig, ax = plt.subplots(figsize=(11.5, 4.8))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.13)
    ax.plot([s[0] for _, s in twin], [s[1] for _, s in twin], color=DIM, ls="--", lw=1.4)
    ax.plot(xs, ys, color=CHALK)
    for t_, s in run[::300]:                                      # velocity arrows every 0.3 s
        ax.annotate("", xy=(s[0] + 0.12 * s[2], s[1] + 0.12 * s[3]), xytext=(s[0], s[1]),
                    arrowprops=dict(arrowstyle="->", color=SKY, lw=1.6))
    ax.scatter([x_end], [0], s=110, color=GOLD, zorder=4)
    ax.set_xlim(0, 95); ax.set_ylim(0, 47); ax.set_xlabel("across (m)"); ax.set_ylabel("height (m)")
    fig.suptitle(f"{speed:g} m/s at {angle:g}°: simulated, lands {x_end:.2f} m away after {t_end:.2f} s, peak "
                 f"{max(ys):.2f} m;  formulas: R = {r_range(speed, angle):.2f} m, T = {t_flight(speed, angle):.2f} s, "
                 f"H = {h_max(speed, angle):.2f} m", x=0.07, ha="left", fontsize=12.5, color=GOLD)
    footnote(fig); plt.show()

for _th in (15, 30, 45, 60, 75):                                   # the course's five jets: simulation = formula, to 1 mm
    assert abs(fly(20, _th)[-1][1][0] - r_range(20, _th)) < 1e-3
assert round(r_range(20, 45), 2) == 40.82 and round(r_range(20, 15), 2) == round(r_range(20, 75), 2) == 20.41

In [ ]:
interact(launch_picture, speed=FloatSlider(value=20, min=5, max=30, step=0.5, description="speed (m/s)"),
         angle=FloatSlider(value=30, min=5, max=85, step=1, description="angle (°)"));

### Push one phone, drop the other

Two phones leave a table 0.8 m high at the same moment: one is pushed across at 1.5 m/s, the other just drops. The
dots are their positions every 0.05 s. Slide the push: the pushed phone lands farther away, but always at the same
moment as the dropped one, t = √(2h / g) = 0.40 s, because the push is across and gravity is down.

In [ ]:
def phones_picture(push=1.5, height=0.8):
    pushed, dropped = fly(push, 0, height), fly(0, 0, height)
    fig, ax = plt.subplots(figsize=(11.5, 4.8))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.86, bottom=0.13)
    ax.plot([-0.4, 0], [height, height], color=DIM, lw=5)
    for run, col in ((dropped, DIM), (pushed, CHALK)):
        ax.plot([s[0] for _, s in run], [s[1] for _, s in run], color=col, lw=1.2)
        pts = run[::50]
        ax.scatter([s[0] for _, s in pts], [s[1] for _, s in pts], s=40, color=col, zorder=3)
    for t_, s in pushed[::50]:
        ax.plot([0, s[0]], [s[1], s[1]], color=GHOST, lw=0.8)
    ax.scatter([pushed[-1][1][0], 0], [0, 0], s=110, color=GOLD, zorder=4)
    ax.set_xlim(-0.45, 2.7); ax.set_ylim(0, 2.1); ax.set_xlabel("across (m)"); ax.set_ylabel("height (m)")
    fig.suptitle(f"both land after {pushed[-1][0]:.3f} s and {dropped[-1][0]:.3f} s (√(2h/g) = "
                 f"{math.sqrt(2 * height / g):.3f} s); the pushed one {pushed[-1][1][0]:.2f} m across",
                 x=0.07, ha="left", fontsize=14, color=GOLD)
    footnote(fig); plt.show()

_p, _d = fly(1.5, 0, 0.8), fly(0, 0, 0.8)
assert close(_p[-1][0], _d[-1][0]) and round(_p[-1][0], 3) == 0.404 and round(_p[-1][1][0], 2) == 0.61

In [ ]:
interact(phones_picture, push=FloatSlider(value=1.5, min=0, max=4, step=0.1, description="push (m/s)"),
         height=FloatSlider(value=0.8, min=0.2, max=2, step=0.1, description="height (m)"));

### Round the roundabout

A car goes round a roundabout of radius 25 m at a steady 10 m/s. The simulation gives it only one thing: an
acceleration v² / r = 4 m/s² pointing at the centre. That alone keeps it on the circle, and it comes back to the
start after T = 2πr / v = 15.71 s. The SKY arrow is its velocity (along the circle); the MINT arrow is its
acceleration (towards the centre). Slide the speed and the radius.

In [ ]:
def roundabout_picture(radius=25.0, speed=10.0):
    run = circle_run(radius, speed)
    drift = max(abs(math.hypot(s[0], s[1]) - radius) for _, s in run)
    k = len(run) // 8; t_k, (x, y, vx, vy) = run[k]
    a = speed ** 2 / radius
    fig, ax = plt.subplots(figsize=(11.5, 5.8))
    fig.subplots_adjust(left=0.05, right=0.98, top=0.87, bottom=0.14)
    ax.plot([s[0] for _, s in run], [s[1] for _, s in run], color=CHALK, lw=1.6)
    ax.scatter([0], [0], s=30, color=DIM)
    ax.scatter([x], [y], s=110, color=GOLD, zorder=4)
    kw = dict(angles="xy", scale_units="xy", scale=1, width=0.006)
    ax.quiver([x], [y], [vx], [vy], color=SKY, **kw)
    ax.quiver([x], [y], [-x / radius * a * 2], [-y / radius * a * 2], color=MINT, **kw)
    ax.set_xlim(-48, 48); ax.set_ylim(-42, 42); ax.set_aspect("equal"); ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
    fig.suptitle(f"r = {radius:g} m, v = {speed:g} m/s: ω = v/r = {speed / radius:.3f} rad/s, T = {2 * math.pi * radius / speed:.2f} s, "
                 f"a = v²/r = {a:.2f} m/s²; the simulated car stays within {drift:.1e} m of the circle",
                 x=0.05, ha="left", fontsize=12.5, color=GOLD)
    footnote(fig, "simulated · checked against the formula · a drawn at 2× scale"); plt.show()

_ring = circle_run(25, 10)
assert max(abs(math.hypot(s[0], s[1]) - 25) for _, s in _ring) < 1e-6 * 25 and close(_ring[-1][0], 2 * math.pi * 25 / 10, rel=1e-6)

In [ ]:
interact(roundabout_picture, radius=FloatSlider(value=25, min=5, max=40, step=1, description="radius (m)"),
         speed=FloatSlider(value=10, min=2, max=20, step=0.5, description="speed (m/s)"));

## Tier 3 · Stretch

Work backwards, spot the trap, combine ideas: the angle from a resultant, a calculator's inverse tangent in the
wrong quadrant, the dot and cross products agreeing, an acceleration from where a drone ends up, the launch speed
for a range, the two angles for one range, a balcony throw (where the range formula is wrong), and the top speed on
a curve.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

**Backwards** problems are the same formulas solved for another letter: the angle from the law of cosines,
u = √(R g / sin 2θ), a = 2(r − u t) / t². **The quadrant trap**: `math.atan(vy / vx)` can't tell (−5, −5) from
(5, 5); `math.atan2(vy, vx)` can (Python: y first). **A launch from a height** isn't a range-formula problem: solve
h + v_y t − ½ g t² = 0 for the landing time first.

In [ ]:
assert agrees("P3.1", [math.degrees(math.acos((5 ** 2 - 7 ** 2 - 4 ** 2) / (2 * 7 * 4)))])        # P3.1: backwards
naive = math.degrees(math.atan(-5 / -5)); true = math.degrees(math.atan2(-5, -5)) % 360           # P3.2: the trap
print(f"P3.2: atan(v y/v x) says {naive:.0f}°, atan2 says {true:.0f}°")
assert round(naive) == 45 and agrees("P3.2", [true])
A, B = (3, 4), (4, 3)                                                                              # P3.3
assert agrees("P3.3", [dot(A, B), cross(A, B), angle_between(A, B)])
assert round(math.degrees(math.asin(abs(cross(A, B)) / (mag(A) * mag(B)))), 9) == round(angle_between(A, B), 9)
assert agrees("P3.4", [2 * (25 - 4 * 5) / 5 ** 2, 2 * (10 - 0 * 5) / 5 ** 2])                     # P3.4: a from r
assert agrees("P3.5", [math.sqrt(50 * g / math.sin(math.radians(60)))])                           # P3.5
low36 = math.degrees(math.asin(12 * g / 12 ** 2)) / 2                                              # P3.6: twins
assert agrees("P3.6", [low36, 90 - low36]) and t_flight(12, low36) < t_flight(12, 90 - low36)
x37, t37 = reach_from_height(15, 35, 6)                                                            # P3.7: a balcony
print(f"P3.7: lands {x37:.2f} m out after {t37:.2f} s; the range formula says {r_range(15, 35):.2f} m")
assert agrees("P3.7", [x37, x37 - r_range(15, 35)]) and close(fly(15, 35, 6)[-1][1][0], x37)
assert agrees("P3.8", [math.sqrt(5 * 80), math.sqrt(5 * 160)])                                     # P3.8: √2, not 2

## Sources
Every exercise and set-up here is our own. The chapter order follows the Class 11
physics syllabus (CBSE Physics 042, 2025–26, chapter 3, "Motion in a Plane"; retrieved 2026-11-11); the definitions
agree with OpenStax, University Physics Volume 1, chapters 2 ("Vectors") and 4 ("Motion in Two and Three
Dimensions") (retrieved 2026-11-11). g = 9.8 m/s² throughout; no air resistance.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/ph03_motion_plane/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("ph03_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"], _q["group"]) == \
               (_e["q"], _e["working"], _e["function"], _e["tier"], _e["group"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and \
               all(abs(a - float(b)) < 1e-9 for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX) and _P.g == g
    assert _P.KICK_RUN[-1] == _kick[-1] and _P.ROUND_RUN[-1] == _ring[-1]          # the same simulator, the same runs
    print("matches the course's practice set")